**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 31: 온톨로지 RAG — RDF · OWL 추론 (rdflib)

## 학습 목표

1️⃣ **온톨로지 = 사실(ABox) + 스키마·규칙(TBox)** 임을 이해한다
2️⃣ **그래프 RAG vs 온톨로지 RAG**의 핵심 차이가 **추론(reasoning)** 임을 안다
3️⃣ `rdflib`로 **RDF 트리플 + RDFS/OWL 스키마**를 구축한다
4️⃣ **SPARQL**로 질의하고, **OWL/RDFS 추론**으로 *명시하지 않은 사실*을 도출한다
5️⃣ **제약(거버넌스 규칙) 검사**와 **온톨로지 RAG + LLM** 연동을 실습한다
6️⃣ 실서비스용 **온톨로지 DB 선택**(Fuseki / GraphDB / Ontop)을 익힌다

---

### ⚠️ 30강(그래프 RAG)과의 관계
- **30강**: Neo4j/NetworkX로 **저장된 관계를 탐색**(traversal) → *그래프 RAG*
- **31강**: 그 위에 **스키마+규칙**을 얹어 *저장 안 한 사실까지 추론* → *온톨로지 RAG*
- 프로퍼티 그래프(Neo4j·Apache AGE)는 OWL 추론이 없어 **그래프 RAG 도구**입니다.
  온톨로지 RAG는 **RDF/OWL 스택**(rdflib → Fuseki/GraphDB)을 씁니다.

### 실습 환경
- **필수 패키지**: `rdflib`, `openai`(선택), `python-dotenv`
- **외부 서비스 불필요** (rdflib 메모리 그래프로 전 과정 실습)

In [1]:
# 패키지 임포트 및 환경 설정
import os
import json
from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL
from urllib.parse import quote
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # OPENAI_API_KEY 등 (있으면 §8에서 사용)

print("rdflib 준비 완료")
print("OPENAI_API_KEY:", "✅" if os.getenv("OPENAI_API_KEY") else "❌ (없으면 §8 LLM 셀은 건너뜀)")

rdflib 준비 완료
OPENAI_API_KEY: ✅


---
## 1️⃣ 그래프 RAG vs 온톨로지 RAG — 핵심은 "추론"

| | 그래프 RAG (30강) | **온톨로지 RAG (31강)** |
|---|---|---|
| 저장 | 사실(노드·엣지) | 사실 + **스키마·규칙(TBox)** |
| 질의 | 저장된 엣지 **탐색** | 저장 안 된 사실도 **추론** |
| DB | 프로퍼티 그래프 (Neo4j·AGE) | **RDF 트리플스토어** (Fuseki·GraphDB) |
| 질의어 | Cypher | **SPARQL** |
| 답하는 것 | "그래프에 **있는** 것" | "그래프에서 **논리적으로 따라 나오는** 것" |

**온톨로지가 추가하는 것 = 추론.** 의학 예:
- `당뇨병 ⊑ 만성질환` → 당뇨병만 저장해도 "만성질환을 보여줘"에 포함 (클래스 계층)
- `약물 상호작용 = 대칭` → 한 방향만 저장해도 양방향 도출 (속성 특성)
- `상호작용 약물 병용` → 위험한 처방 자동 감지 (제약)

---
## 2️⃣ RDF 기초 — 모든 지식을 트리플로

**RDF**는 지식을 `(주어, 술어, 목적어)` **트리플**로 표현합니다.

```
(제2형당뇨병) --[hasSymptom]--> (다뇨)
   주어              술어          목적어
```

- **TBox** (스키마/용어): 클래스·관계의 정의와 규칙 — "당뇨병은 만성질환의 한 종류"
- **ABox** (사실/단언): 실제 인스턴스 — "제2형당뇨병의 증상은 다뇨"

아래 작은 예제로 트리플이 어떻게 생겼는지 봅니다.

In [ ]:
# RDF 첫걸음 — 트리플 3개
EX = Namespace("http://example.org/")
demo = Graph(); demo.bind("ex", EX)

demo.add((EX["제2형당뇨병"], RDF.type,   EX.Diabetes))            # 제2형당뇨병 은 당뇨병(Diabetes) 이다
demo.add((EX["제2형당뇨병"], RDFS.label, Literal("제2형당뇨병")))   # 사람이 읽는 이름
demo.add((EX["제2형당뇨병"], EX.hasSymptom, EX["다뇨"]))           # 제2형당뇨병 --hasSymptom--> 다뇨

print(f"트리플 {len(demo)}개 — Turtle 직렬화:")
print(demo.serialize(format="turtle"))

---
### 🤖 보너스 — LLM 으로 자연어 문장 → ABox + TBox 자동 생성

위 §2 와 아래 §3~§4 에서는 트리플을 **사람이 손으로** 작성합니다. 하지만 실무에서는
진료기록·논문·약품설명서 같은 **자연어 문서**에서 트리플을 뽑아야 합니다. `OPENAI_API_KEY` 가 있으면
LLM 이 한 문장에서 **사실(ABox)** 과 **스키마·규칙(TBox)** 을 구분해 추출할 수 있습니다.

| 입력 문장 | 추출 종류 | 트리플 |
|---|---|---|
| "제2형당뇨병의 증상은 다뇨다" | **ABox** | `제2형당뇨병 hasSymptom 다뇨`, `제2형당뇨병 a Diabetes` |
| "당뇨병은 만성질환의 한 종류다" | **TBox** | `Diabetes rdfs:subClassOf ChronicDisease` |
| "약물 상호작용은 대칭이다" | **TBox** | `interactsWith a owl:SymmetricProperty` |

추출한 트리플을 `rdflib` 그래프로 적재하면, 그 위에서 **§6 의 추론기**가 그대로 동작합니다.

> ⚠️ LLM 은 **사실(ABox) 추출**엔 강하지만, **스키마 설계(TBox)** — 어떤 속성을 대칭/전이로
> 둘지 — 는 도메인(의학) 판단이 필요하므로 전문가가 검수하는 것이 안전합니다.

In [ ]:
# 🤖 LLM 으로 '자유 형식 문서' → ABox + TBox 트리플 자동 추출 (OPENAI_API_KEY 있을 때만)
# ↓ 이 document 를 당신의 진료기록·논문·약품설명서 등 아무 문서로 바꿔보세요.
document = """\
65세 남성 환자가 협심증으로 내원하였다. 협심증은 심장질환의 일종이며, 주요 증상으로
흉통과 호흡곤란이 나타난다. 치료를 위해 니트로글리세린과 아스피린을 처방하였다.
다만 아스피린은 와파린과 상호작용하여 출혈 위험을 높이므로 병용에 주의해야 한다.
약물 상호작용은 양방향(대칭) 관계이다. 한편 이 환자는 제2형당뇨병도 동반하고 있는데,
당뇨병은 만성질환에 속하며 메트포르민으로 관리한다.
"""

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY 없음 — 이 보너스 셀은 건너뜁니다.")
else:
    from openai import OpenAI
    client = OpenAI()

    SYSTEM = """당신은 의료 온톨로지 엔지니어입니다. 주어진 한국어 문서에서 RDF 트리플을 추출해 JSON 으로만 답하세요.
문서에 등장하는 개념에 맞춰 클래스·속성 이름을 스스로 정의하고, 사실(ABox)과 스키마·규칙(TBox)을 구분합니다.

[tbox] 클래스·속성의 스키마/규칙. 아래 4가지 형태만 사용:
  ["<클래스>", "rdfs:subClassOf", "<상위클래스>"]      # "A 는 B 의 일종" → 클래스 계층
  ["<속성>", "rdf:type", "owl:SymmetricProperty"]       # "양방향/대칭" 관계 (예: 약물 상호작용)
  ["<속성>", "rdf:type", "owl:TransitiveProperty"]      # 전이 관계
  ["<속성A>", "owl:inverseOf", "<속성B>"]                # 역관계 (예: 치료 ↔ 치료받음)

[abox] 개체에 대한 사실. 아래 2가지 형태만 사용:
  ["<개체>", "rdf:type", "<클래스>"]                     # 예: 제2형당뇨병 a Diabetes
  ["<개체>", "<속성>", "<개체>"]                         # 예: 협심증 hasSymptom 흉통

규칙:
- 클래스명은 영문 CamelCase(Disease, HeartDisease, Drug, Symptom 등), 속성명은 영문 lowerCamelCase(hasSymptom, treatedBy, interactsWith 등) 로 적절히 만든다.
- 개체(인스턴스)는 문서의 한국어 표현 그대로 사용한다(협심증, 흉통, 아스피린, 와파린).
- 문서에 명시된 내용만 추출하고, 추측하지 않는다.
- 속성의 대칭/전이/역관계 특성은 문서에 그 성질이 **명시적으로 서술된 경우에만** 추출한다.
  (예: "상호작용은 양방향이다" → SymmetricProperty). 속성이 등장했다는 이유만으로 특성을 부여하지 않는다.
- 출력은 {"tbox": [...], "abox": [...]} JSON 하나만."""

    resp = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": document}])
    extracted = json.loads(resp.choices[0].message.content)

    print("── LLM 추출 결과 ──")
    print("TBox (스키마·규칙):")
    for t in extracted.get("tbox", []): print("   ", t)
    print("ABox (사실):")
    for t in extracted.get("abox", []): print("   ", t)

    # ── 추출한 트리플 → rdflib 그래프로 적재 ──
    LX = Namespace("http://example.org/med-llm#")
    gl = Graph(); gl.bind("ex", LX); gl.bind("owl", OWL); gl.bind("rdfs", RDFS)
    PREDEF = {"rdf:type": RDF.type, "rdfs:subClassOf": RDFS.subClassOf,
              "rdfs:label": RDFS.label, "owl:inverseOf": OWL.inverseOf,
              "owl:SymmetricProperty": OWL.SymmetricProperty,
              "owl:TransitiveProperty": OWL.TransitiveProperty, "owl:Class": OWL.Class}

    def term(tok):
        """문자열 토큰 → rdflib 용어 (접두사는 표준 어휘로, 나머지는 ex: URI)."""
        tok = tok.strip()
        return PREDEF.get(tok, LX[quote(tok)])

    for triple in extracted.get("tbox", []) + extracted.get("abox", []):
        if len(triple) != 3:
            continue                                   # 형식이 어긋난 행은 건너뜀
        s, p, o = triple
        gl.add((term(s), term(p), term(o)))
        # 한국어가 포함된 개체에는 사람이 읽는 라벨(rdfs:label)도 부여 (클래스/속성은 영문이라 제외)
        for x in (s, o):
            if x not in PREDEF and not x.isascii():
                gl.add((term(x), RDFS.label, Literal(x)))

    print(f"\n── rdflib 적재: 트리플 {len(gl)}개 — Turtle ──")
    print(gl.serialize(format="turtle"))
    print("→ 자유 형식 문서 한 편에서 ABox(사실)와 TBox(규칙)가 함께 추출됩니다.")
    print("  이 그래프에 §6 의 reason() 을 돌리면 대칭·계층 추론으로 사실이 더 늘어납니다.")

---
## 3️⃣ 도메인 데이터 — 의학 (질환·약물·증상·전문의)

질환·증상·치료약·전문의, 그리고 **약물 상호작용** 관계를 사실(ABox)로 준비합니다.
멀티홉 그래프 위에 *추론을 더했을 때 무엇이 달라지나* 를 보기 위함입니다.

In [ ]:
medical_data = {
    "nodes": [
        # 질환 — subtype 으로 클래스 계층 시연 (당뇨병 ⊑ 만성질환 ⊑ 질환)
        {"label": "Disease", "props": {"name": "제2형당뇨병",  "subtype": "당뇨병"}},
        {"label": "Disease", "props": {"name": "임신성당뇨병", "subtype": "당뇨병"}},
        {"label": "Disease", "props": {"name": "고혈압",       "subtype": "만성질환"}},
        {"label": "Disease", "props": {"name": "협심증",       "subtype": "심장질환"}},
        # 약물 — subtype = 약물 계열
        {"label": "Drug", "props": {"name": "메트포르민",     "subtype": "당뇨약"}},
        {"label": "Drug", "props": {"name": "인슐린",         "subtype": "당뇨약"}},
        {"label": "Drug", "props": {"name": "암로디핀",       "subtype": "혈압약"}},
        {"label": "Drug", "props": {"name": "니트로글리세린", "subtype": "혈관확장제"}},
        {"label": "Drug", "props": {"name": "아스피린",       "subtype": "항혈소판제"}},
        {"label": "Drug", "props": {"name": "와파린",         "subtype": "항응고제"}},
        # 증상
        {"label": "Symptom", "props": {"name": "다뇨"}},
        {"label": "Symptom", "props": {"name": "갈증"}},
        {"label": "Symptom", "props": {"name": "두통"}},
        {"label": "Symptom", "props": {"name": "흉통"}},
        # 전문의 — 전문과 → subClassOf 시연 (내분비내과의/심장내과의 ⊑ 의사)
        {"label": "Doctor", "props": {"name": "김내분", "role": "내분비내과"}},
        {"label": "Doctor", "props": {"name": "이심장", "role": "심장내과"}},
    ],
    "edges": [
        # 질환 → 증상
        {"from": "제2형당뇨병", "rel": "HAS_SYMPTOM", "to": "다뇨"},
        {"from": "제2형당뇨병", "rel": "HAS_SYMPTOM", "to": "갈증"},
        {"from": "고혈압",     "rel": "HAS_SYMPTOM", "to": "두통"},
        {"from": "협심증",     "rel": "HAS_SYMPTOM", "to": "흉통"},
        # 질환 → 치료약
        {"from": "제2형당뇨병", "rel": "TREATED_BY", "to": "메트포르민"},
        {"from": "제2형당뇨병", "rel": "TREATED_BY", "to": "인슐린"},
        {"from": "고혈압",     "rel": "TREATED_BY", "to": "암로디핀"},
        {"from": "협심증",     "rel": "TREATED_BY", "to": "니트로글리세린"},
        {"from": "협심증",     "rel": "TREATED_BY", "to": "아스피린"},
        # 전문의 → 전문 질환
        {"from": "김내분", "rel": "SPECIALIZES_IN", "to": "제2형당뇨병"},
        {"from": "김내분", "rel": "SPECIALIZES_IN", "to": "고혈압"},
        {"from": "이심장", "rel": "SPECIALIZES_IN", "to": "협심증"},
        # 약물 상호작용 — '한 방향만' 저장 (대칭 추론으로 반대 방향을 도출할 것)
        {"from": "아스피린", "rel": "INTERACTS_WITH", "to": "와파린"},
    ],
}
print(f"노드 {len(medical_data['nodes'])}개, 엣지 {len(medical_data['edges'])}개 준비")

---
## 4️⃣ rdflib 로 온톨로지 구축 — TBox(스키마) + ABox(사실)

- **TBox**: 클래스 계층(`rdfs:subClassOf`: 당뇨병⊑만성질환⊑질환), 속성 특성(`owl:SymmetricProperty`: 상호작용, `owl:inverseOf`: 치료↔치료받음)
- **ABox**: `medical_data` 의 사실을 트리플로 변환

In [ ]:
MED = Namespace("http://example.org/med#")
g = Graph(); g.bind("med", MED); g.bind("owl", OWL)
def uri(name): return MED[quote(name)]

# ── TBox: 스키마와 규칙 ──
for c in ["Disease", "Drug", "Symptom", "Doctor"]:
    g.add((MED[c], RDF.type, OWL.Class))
# 질환 계층
g.add((MED.ChronicDisease, RDFS.subClassOf, MED.Disease))         # 만성질환 ⊑ 질환
g.add((MED.HeartDisease,   RDFS.subClassOf, MED.Disease))         # 심장질환 ⊑ 질환
g.add((MED.Diabetes,       RDFS.subClassOf, MED.ChronicDisease))  # 당뇨병 ⊑ 만성질환 (전이 시연)
# 약물 계열 ⊑ 약물
for c in ["DiabetesDrug", "BPDrug", "Vasodilator", "Antiplatelet", "Anticoagulant"]:
    g.add((MED[c], RDFS.subClassOf, MED.Drug))
# 전문의 ⊑ 의사
g.add((MED.Endocrinologist, RDFS.subClassOf, MED.Doctor))   # 내분비내과의 ⊑ 의사
g.add((MED.Cardiologist,    RDFS.subClassOf, MED.Doctor))   # 심장내과의 ⊑ 의사
# 속성 특성
g.add((MED.interactsWith, RDF.type, OWL.SymmetricProperty)) # 약물 상호작용 = 대칭
g.add((MED.treatedBy, OWL.inverseOf, MED.treats))           # 치료받음 의 역관계 = 치료
g.add((MED.hasSymptom, OWL.inverseOf, MED.symptomOf))       # 증상보유 의 역관계 = 증상소속

REL = {"HAS_SYMPTOM": MED.hasSymptom, "TREATED_BY": MED.treatedBy,
       "SPECIALIZES_IN": MED.specializesIn, "INTERACTS_WITH": MED.interactsWith}
SUBTYPE = {"당뇨병": MED.Diabetes, "만성질환": MED.ChronicDisease, "심장질환": MED.HeartDisease,
           "당뇨약": MED.DiabetesDrug, "혈압약": MED.BPDrug, "혈관확장제": MED.Vasodilator,
           "항혈소판제": MED.Antiplatelet, "항응고제": MED.Anticoagulant}
ROLE = {"내분비내과": MED.Endocrinologist, "심장내과": MED.Cardiologist}

# ── ABox: 사실 ──
known = set()
for node in medical_data["nodes"]:
    nm = node["props"]["name"]; known.add(nm)
    lbl = node["label"]
    if lbl in ("Disease", "Drug"):
        cls = SUBTYPE.get(node["props"].get("subtype"), MED[lbl])
    elif lbl == "Doctor":
        cls = ROLE.get(node["props"].get("role"), MED.Doctor)
    else:
        cls = MED.Symptom
    g.add((uri(nm), RDF.type, cls)); g.add((uri(nm), RDFS.label, Literal(nm)))

for e in medical_data["edges"]:
    for ep in (e["from"], e["to"]):
        if ep not in known:
            g.add((uri(ep), RDFS.label, Literal(ep))); known.add(ep)
    g.add((uri(e["from"]), REL[e["rel"]], uri(e["to"])))

print(f"RDF 트리플 {len(g)}개 구축\n")
print("Turtle 미리보기 (일부):")
print("\n".join(g.serialize(format="turtle").splitlines()[3:13]))

---
### 🛠️ 실습 도우미 — 결과를 **표**·**그림**으로 보기

SPARQL 결과를 pandas **표**(DataFrame)로, 온톨로지를 networkx **그래프 그림**으로 봅니다.
(설치: `pip install pandas matplotlib networkx`)

In [ ]:
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import networkx as nx
matplotlib.rcParams["font.family"] = "NanumGothic"      # 한글 라벨
matplotlib.rcParams["axes.unicode_minus"] = False

def sparql_df(query, graph=g):
    """SPARQL 결과 → pandas DataFrame (셀 마지막 줄에 두면 표로 렌더)."""
    res = graph.query(query)
    return pd.DataFrame([{str(v): str(r[v]) for v in res.vars} for r in res])

_REL_KO = {MED.hasSymptom:"증상", MED.treatedBy:"치료약", MED.specializesIn:"전문",
           MED.interactsWith:"상호작용", MED.treats:"치료", MED.symptomOf:"증상소속"}

def draw_ontology(graph, title="온톨로지 그래프"):
    """인스턴스 관계를 그래프로 시각화 (rdfs:label 사용)."""
    lbl = {s:str(o) for s,_,o in graph.triples((None, RDFS.label, None))}
    G = nx.DiGraph()
    for prop, ko in _REL_KO.items():
        for s,_,o in graph.triples((None, prop, None)):
            if s in lbl and o in lbl: G.add_edge(lbl[s], lbl[o], label=ko)
    plt.figure(figsize=(13,9))
    pos = nx.spring_layout(G, k=0.9, seed=42)
    nx.draw(G, pos, with_labels=True, node_color="#9ec5fe", node_size=1400,
            font_size=9, font_family="NanumGothic", edge_color="gray", arrows=True, arrowsize=15)
    nx.draw_networkx_edge_labels(G, pos, font_family="NanumGothic", font_size=7,
            edge_labels=nx.get_edge_attributes(G, "label"))
    plt.title(title, fontsize=14); plt.axis("off"); plt.tight_layout(); plt.show()

print("헬퍼 준비 완료: sparql_df(query) / draw_ontology(graph)")

In [ ]:
# 온톨로지 그래프 시각화 (인스턴스 관계)
draw_ontology(g, title="의학 온톨로지 — 질환·약물·증상·전문의")

---
## 5️⃣ SPARQL 질의

SPARQL 은 그래프용 질의어입니다. 멀티홉 탐색 자체는 **그래프 RAG(30강)와 공유**하는 능력이며, 31강의 차별점은 **다음 절의 추론**입니다.

In [ ]:
PFX = """PREFIX med: <http://example.org/med#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>"""

print("[Q1] 질환별 증상")
for r in g.query(PFX + " SELECT ?dis ?sym WHERE { ?d med:hasSymptom ?s . ?d rdfs:label ?dis . ?s rdfs:label ?sym } ORDER BY ?dis"):
    print(f"  - {r.dis} → {r.sym}")

print("\n[Q2] 3-hop: 김내분 의사가 전문으로 보는 질환을 치료하는 약물 (그래프 RAG 와 공유)")
q2 = PFX + """
SELECT ?disease ?drug WHERE {
  ?doc rdfs:label "김내분" ; med:specializesIn ?dis .
  ?dis rdfs:label ?disease ; med:treatedBy ?d .
  ?d rdfs:label ?drug .
} ORDER BY ?disease ?drug"""
for r in g.query(q2):
    print(f"   {r.disease} → {r.drug}")

In [ ]:
# 같은 3-hop 결과를 '표'로 보기 (sparql_df 헬퍼)
df = sparql_df(q2)
print(df.to_string(index=False))
df  # Jupyter 에서 HTML 표로 렌더

---
## 6️⃣ OWL / RDFS 추론 — ★ 온톨로지 RAG의 핵심

**명시하지 않은 사실**을 규칙으로 자동 도출합니다. 그래프 RAG(30강)는 못 하는 부분.

적용 규칙(전방향 추론):
- **RDFS**: `(s a C) ∧ (C ⊑ D) ⟹ (s a D)`  / `subClassOf` 전이
- **OWL**: `SymmetricProperty`(대칭), `inverseOf`(역관계), `TransitiveProperty`(전이)

In [ ]:
def reason(graph):
    """간단한 전방향 추론기 (RDFS subClassOf + OWL Symmetric/Inverse/Transitive)."""
    base = len(graph); changed = True
    while changed:
        changed = False; new = []
        # RDFS: 타입 전파 + subClassOf 전이
        for s, _, c in graph.triples((None, RDF.type, None)):
            for _, _, d in graph.triples((c, RDFS.subClassOf, None)):
                new.append((s, RDF.type, d))
        for c, _, d in graph.triples((None, RDFS.subClassOf, None)):
            for _, _, e in graph.triples((d, RDFS.subClassOf, None)):
                new.append((c, RDFS.subClassOf, e))
        # OWL 대칭
        for p, _, _ in graph.triples((None, RDF.type, OWL.SymmetricProperty)):
            for x, _, y in graph.triples((None, p, None)):
                new.append((y, p, x))
        # OWL 역관계
        for p, _, q in graph.triples((None, OWL.inverseOf, None)):
            for x, _, y in graph.triples((None, p, None)):
                new.append((y, q, x))
        # OWL 전이
        for p, _, _ in graph.triples((None, RDF.type, OWL.TransitiveProperty)):
            for x, _, y in graph.triples((None, p, None)):
                for _, _, z in graph.triples((y, p, None)):
                    new.append((x, p, z))
        for t in new:
            if t not in graph:
                graph.add(t); changed = True
    return len(graph) - base

q_doctor = PFX + " SELECT ?n WHERE { ?p a med:Doctor ; rdfs:label ?n }"
before = {str(r.n) for r in g.query(q_doctor)}
print(f"[추론 전] med:Doctor 직접 인스턴스: {len(before)}명  (전문의는 Doctor 로 안 잡힘)")

added = reason(g)
after = {str(r.n) for r in g.query(q_doctor)}
print(f"[추론] 새 트리플 {added}개 생성")
print(f"[추론 후] med:Doctor: {len(after)}명 → subClassOf 로 추론: {sorted(after - before)}")

# 클래스 계층 — 당뇨병 ⊑ 만성질환 ⊑ 질환
print("\n[계층] 모든 만성질환 (당뇨병도 subClassOf 로 포함):")
for r in g.query(PFX + " SELECT ?n WHERE { ?d a med:ChronicDisease ; rdfs:label ?n }"):
    print("  -", str(r.n))

# 대칭 추론
print("\n[대칭] interactsWith — 한 방향만 줬는데 양방향 도출:")
for r in g.query(PFX + " SELECT ?a ?b WHERE { ?x med:interactsWith ?y . ?x rdfs:label ?a . ?y rdfs:label ?b }"):
    print(f"   {r.a} <-> {r.b}")

# 역관계 추론
print("\n[역관계] treatedBy 의 역관계 treats 자동 생성 (예시 3개):")
for r in list(g.query(PFX + " SELECT ?drug ?dis WHERE { ?d med:treats ?x . ?d rdfs:label ?drug . ?x rdfs:label ?dis }"))[:3]:
    print(f"   {r.drug} 는 {r.dis} 를 치료")

---
## 7️⃣ 제약 / 일관성 검사 — 의료 안전 규칙

온톨로지는 "있으면 안 될" 상태를 **규칙으로 검사**할 수 있습니다.
예: **병용 금기** — 한 질환의 치료약 중 서로 **상호작용**하는 약이 함께 있으면 위험.

In [ ]:
def check_dangerous_combo(graph):
    q = PFX + """
    SELECT ?disease ?d1 ?d2 WHERE {
      ?dis med:treatedBy ?x , ?y .
      ?x med:interactsWith ?y .
      ?dis rdfs:label ?disease . ?x rdfs:label ?d1 . ?y rdfs:label ?d2 .
      FILTER(STR(?d1) < STR(?d2))
    }"""
    return [(str(r.disease), str(r.d1), str(r.d2)) for r in graph.query(q)]

print("[정상 상태] 병용 금기 검사:", check_dangerous_combo(g) or "위반 없음 ✅")

# 위반 시나리오: 협심증 치료약에 와파린을 추가 (이미 치료약인 아스피린과 상호작용)
g.add((uri("협심증"), MED.treatedBy, uri("와파린")))
print("[위반 추가] 협심증 치료약에 와파린 추가 →", check_dangerous_combo(g))
g.remove((uri("협심증"), MED.treatedBy, uri("와파린")))   # 원복
print("[원복 후]", check_dangerous_combo(g) or "위반 없음 ✅")

---
## 8️⃣ 온톨로지 RAG + LLM

**추론된 그래프**에서 SPARQL 로 사실을 검색(retrieve)하고, 그 사실을 LLM 에 넘겨 자연어 답을 생성합니다.
포인트: 검색 대상이 *추론으로 보강된* 그래프라, 그래프 RAG 가 놓치는 사실까지 답합니다.

In [ ]:
question = "와파린과 함께 복용하면 위험한 약은 무엇인가요?"

# 1) 추론된 그래프에서 관련 사실 검색 (대칭 추론 덕에 양방향 모두 잡힘)
facts = [f"{r.a} interactsWith {r.b}" for r in g.query(
    PFX + " SELECT ?a ?b WHERE { ?x med:interactsWith ?y . ?x rdfs:label ?a . ?y rdfs:label ?b }")]
retrieved = [f for f in facts if "와파린" in f]
print("검색된 사실:", retrieved)

# 2) LLM 으로 자연어 답변 (키 있을 때만)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    from openai import OpenAI
    client = OpenAI()
    prompt = f"다음 사실만 근거로 질문에 한국어로 간결히 답하세요.\n사실: {retrieved}\n질문: {question}"
    resp = client.chat.completions.create(model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}], temperature=0)
    print("\n[LLM 답변]", resp.choices[0].message.content)
else:
    print("\n(OPENAI_API_KEY 없음 — 검색된 사실로 답: 와파린은 아스피린과 상호작용)")

---
## 9️⃣ 그래프 RAG vs 온톨로지 RAG — 같은 질문, 다른 답

"와파린과 상호작용하는 약은?" — 데이터에는 `아스피린 →interactsWith→ 와파린` **한 방향만** 저장돼 있습니다.

In [ ]:
mini = Graph(); mini.bind("med", MED)
mini.add((MED.interactsWith, RDF.type, OWL.SymmetricProperty))
mini.add((uri("아스피린"), MED.interactsWith, uri("와파린")))  # 한 방향만
mini.add((uri("와파린"), RDFS.label, Literal("와파린")))
mini.add((uri("아스피린"), RDFS.label, Literal("아스피린")))

q = PFX + """ SELECT ?n WHERE {
  ?s rdfs:label "와파린" . ?s med:interactsWith ?x . ?x rdfs:label ?n }"""

print("질문: 와파린과 상호작용하는 약은?")
print("  [그래프 RAG] 저장된 방향만 탐색 →", [str(r.n) for r in mini.query(q)] or "(없음)")
reason(mini)
print("  [온톨로지 RAG] 대칭 추론 후     →", [str(r.n) for r in mini.query(q)])
print("\n→ 같은 데이터·같은 질문인데, 추론이 있으면 답이 달라진다 (이것이 온톨로지 RAG)")

---
## 9️⃣.5 실제 오픈소스 RDF DB 로 SPARQL — **Oxigraph**

지금까지는 rdflib **메모리** 그래프였습니다. 실서비스처럼 **디스크에 영속되는
오픈소스 트리플스토어**에 적재해 SPARQL 로 질의합니다.

- **Oxigraph** = Rust 기반 임베디드 RDF DB (`pip install pyoxigraph`) — **Java·서버 불필요**
- 패턴: rdflib 에서 **추론까지 끝낸 트리플**을 DB 에 적재(materialize) → SPARQL 서비스
- **Jena Fuseki** 는 같은 일을 **HTTP SPARQL 엔드포인트**로 제공(Java 필요).
  GraphDB/Stardog 은 적재 시 OWL 추론을 DB 가 자동 수행.

In [ ]:
from pyoxigraph import Store, RdfFormat
import shutil

DB_PATH = "oxigraph_db"
shutil.rmtree(DB_PATH, ignore_errors=True)
store = Store(DB_PATH)                                   # 디스크 영속 트리플스토어
store.load(g.serialize(format="nt").encode("utf-8"),    # 추론까지 끝낸 rdflib 그래프를 적재
           format=RdfFormat.N_TRIPLES)
print(f"Oxigraph 적재 완료 — 트리플 {len(store)}개 (추론분 포함)\n")

PFX_S = """PREFIX med: <http://example.org/med#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>"""

print("[Oxigraph SPARQL] 김내분 의사 전문 질환의 치료약:")
q = PFX_S + """ SELECT ?disease ?drug WHERE {
  ?doc rdfs:label "김내분" ; med:specializesIn ?dis .
  ?dis rdfs:label ?disease ; med:treatedBy ?d . ?d rdfs:label ?drug .
} ORDER BY ?disease ?drug"""
for sol in store.query(q):
    print(f"   {sol['disease'].value} → {sol['drug'].value}")

print("\n[Oxigraph SPARQL] 와파린 상호작용 약물 (대칭 추론분도 DB 에 적재됨):")
for sol in store.query(PFX_S + """ SELECT ?n WHERE {
  ?s rdfs:label "와파린" . ?s med:interactsWith ?x . ?x rdfs:label ?n }"""):
    print("  -", sol["n"].value)

del store                                               # 닫기
store2 = Store(DB_PATH)                                  # 재오픈 → 영속 확인
print(f"\n재오픈 후 트리플 {len(store2)}개 — 디스크에 영속됨 ✅")
del store2
shutil.rmtree(DB_PATH, ignore_errors=True)              # 실습 후 정리

---
## 🔟 실서비스: 온톨로지 RAG 용 DB 선택

메모리 rdflib 는 실습/프로토타입용입니다. 프로덕션은 **OWL 추론을 지원하는 RDF 트리플스토어**가 필요합니다.

| 상황 | DB | 비고 |
|---|---|---|
| 표준/오픈소스 | **Apache Jena + Fuseki** | RDF·SPARQL·추론, SPARQL 엔드포인트 |
| 프로덕션 추론 | **GraphDB / Stardog** | OWL RL 추론 강력 |
| 기존 관계형 DB에 온톨로지 | **Ontop + PostgreSQL** | **OWL QL**(질의 재작성, OBDA) |
| 관리형 클라우드 | **Amazon Neptune** | RDF/SPARQL (추론은 제한적) |
| 실습/프로토타입 | **rdflib (+owlrl)** | 본 노트북 |

> ❌ **Neo4j · Apache AGE** = 프로퍼티 그래프 → OWL 추론 없음 → *그래프 RAG(30강)용*. 온톨로지 RAG 에는 부적합.

In [ ]:
# (1) 메모리 그래프를 파일로 영속화 — 표준 RDF 포맷(Turtle)
g.serialize(destination="medical_ontology.ttl", format="turtle")
print("저장: medical_ontology.ttl  (트리플", len(g), "개)")

# (2) 프로덕션 트리플스토어(Fuseki/GraphDB)에 올리고 SPARQL 엔드포인트로 질의 — 개념 코드
print("""
# ── Apache Jena Fuseki 예시 (개념) ─────────────────────────
#   docker run -p 3030:3030 stain/jena-fuseki        # Fuseki 기동
#   # 데이터셋 생성 후 medical_ontology.ttl 업로드
#
#   from SPARQLWrapper import SPARQLWrapper, JSON
#   sparql = SPARQLWrapper("http://localhost:3030/med/sparql")
#   sparql.setQuery(PFX + " SELECT ?n WHERE { ?x a med:Disease ; rdfs:label ?n }")
#   sparql.setReturnFormat(JSON)
#   results = sparql.query().convert()
#
# GraphDB/Stardog 는 OWL 추론을 DB 차원에서 자동 수행(메모리 추론기 불필요).
# Ontop 은 PostgreSQL 위에서 SPARQL→SQL 재작성(OWL QL).
""")

---
## 📝 정리

1. **온톨로지 = 사실(ABox) + 스키마·규칙(TBox)** → 명시 안 한 사실을 **추론**
2. **그래프 RAG(30) vs 온톨로지 RAG(31)** 의 차이는 DB가 아니라 **추론 계층**
   - 멀티홉 탐색은 둘 다 가능(겹침) / **추론·제약 검사는 온톨로지 RAG 만**
3. `rdflib` 로 **RDF + RDFS/OWL** 구축, **SPARQL** 질의, **전방향 추론**
   - subClassOf(당뇨병⊑만성질환), SymmetricProperty(약물 상호작용), inverseOf(치료↔치료받음)
4. **제약 검사**(병용 금기)로 처방 안전성 검증
5. **온톨로지 RAG + LLM**: *추론으로 보강된* 그래프를 검색해 LLM 답변
6. 실서비스 DB: **Fuseki / GraphDB / Ontop** (RDF/OWL), *Neo4j·AGE 아님*

### 다음 단계
- owlrl 등 완전한 OWL 추론기 적용
- Fuseki/GraphDB 에 적재 후 SPARQL 엔드포인트 서비스화
- 표준 의료 온톨로지 연계 (SNOMED CT · ICD-11 · RxNorm)